In [26]:
unquote_col = ["race", "race_o", "field"]
encoding_cols = ["gender", "race", "race_o", "field"]
preferences = ["attractive_important", "sincere_important", "intelligence_important", 
            "funny_important", "ambition_important", "shared_interests_important"]

preferences_partner = ["pref_o_attractive", "pref_o_sincere", "pref_o_intelligence", "pref_o_funny", "pref_o_ambitious",
                    "pref_o_shared_interests"]

rating_of_partner = ["attractive_partner", "sincere_partner", "intelligence_partner", "funny_partner",
                    "ambition_partner", "shared_interests_partner"]

continuos_valued = ["age","age_o","importance_same_race","importance_same_religion","pref_o_attractive",
                    "pref_o_sincere","pref_o_intelligence","pref_o_funny","pref_o_ambitious","pref_o_shared_interests",
                    "attractive_important","sincere_important","intelligence_important","funny_important",
                    "ambition_important", "shared_interests_important","attractive","sincere","intelligence","funny",
                    "ambition","attractive_partner","sincere_partner","intelligence_partner","funny_partner",
                    "ambition_partner","shared_interests_partner","sports","tvsports","exercise","dining","museums",
                    "art","hiking","gaming","clubbing","reading","tv","theater","movies","concerts","music","shopping",
                    "yoga","interests_correlate","expected_happy_with_sd_people","like"]

maxval_ten = [ "importance_same_race","importance_same_religion", "attractive","sincere","intelligence","funny",
                    "ambition","attractive_partner","sincere_partner","intelligence_partner","funny_partner",
                    "ambition_partner","shared_interests_partner","sports","tvsports","exercise","dining","museums",
                    "art","hiking","gaming","clubbing","reading","tv","theater","movies","concerts","music","shopping",
                    "yoga", "like", "expected_happy_with_sd_people"]


import csv

from statistics import mean
import numpy as np
import matplotlib.pyplot as plt


# Performs label encoding in the database
# input:  the whole database already preprocessed
# output: a dictionary of encodings, indexed by each __encoding_cols__
# each index of the dictionary is lexicographically ordered, therefore the label the position of the value in this list
def label_encode(dic, enc_cols):
    # initialize the encoding dictionary
    enc = {}
    for enc_col in enc_cols:
        # we use sets for the label encoding, this way we always only add one element
        # each column in encoding_cols have a different set of label encodings
        enc[enc_col] = set()

    for row in dic:
        for enc_col in enc_cols:
            enc[enc_col].add(row[enc_col])
    
    for enc_col in enc_cols:
        enc[enc_col] = list(enc[enc_col])
        enc[enc_col].sort()

    return enc


def get_encoding(label, attr, encodings):
    for id, _label in enumerate(encodings[attr]):
        if label == _label:
            return id

    raise Exception

def encode_dic(dic, enc_cols, encodings):
    for row in dic:
        for attr in enc_cols:
            enc = get_encoding(row[attr], attr, encodings)
            row[attr] = enc
    
    return dic
            


def normalize(dic, prefs):
    total = 0

    # calculate total
    for row in dic:
        for preference in prefs:
            total += float(row[preference])

        # normalize
        for preference in prefs:
            row[preference] = float(row[preference]) / total
        total = 0
    

    return dic

# Unquotes the columns defined in __unquote_col__
# input: a row of dating-full in the form of dictionary
# output: (count, row)
#       count: number of unquoted cells
#       row: post processesd quote
def unquote(row):
    count = 0
    for col in unquote_col:
        cell = row[col]
        if cell.startswith("'") and cell.endswith("'") and len(cell) > 1:
            cell = cell.strip("'")
            count += 1
        if cell.startswith('"') and cell.endswith('"') and len(cell) > 1:
            cell = cell.strip('"')
            count += 1
        row[col] = cell
    return (count, row)

# Lowercases the __field__ column
# input: a row of dating-full in the form of a dictionary
# output: (count, row)
#       count: number of 
def tolower_field(row):
    count = 0
    if(not row["field"].islower()):
        count += 1
        row["field"] = row["field"].lower()
    return(count, row)

# Processes a dictionary by doing the following operations:
# 1 - unquote cells in unquote_col
# 2 - convert values in column field to lowercase
# input: csvDicReader
# output: (lower_count count, encodings, out_dic)
#       unq_count: number of unquoted cells
#       unq_count: number of lower cased field cells
#       label_encodings: encodings for the columns in encodings_val
#       outdic: preprocessed dictionary 
def preprocess(in_dic):
    unq_count = 0
    lower_count = 0
    out_dic = []
    for row in in_dic:
        c, row = unquote(row)
        c1, row = tolower_field(row)
        unq_count += c
        lower_count += c1
        out_dic.append(row)
    
    out_dic = normalize(out_dic, preferences)
    out_dic = normalize(out_dic, preferences_partner)
    # out_dic = encode_dic(out_dic, encoding_cols, label_encodings)
    #label_encodings = label_encode(out_dic, encoding_cols)
    return (unq_count, lower_count, out_dic)

def print_encodings(enc):
    for enc_col in encoding_cols:
        for id, val in enumerate(enc[enc_col]):
            if val == 'male' or (val == 'European/Caucasian-American' and enc_col == 'race') or (val == 'Latino/Hispanic American' and enc_col == 'race_o') or val == 'law':
                print("Value assigned for {} in column {}: {}".format(val, enc_col, id))
            

def print_means(dic):
    for pref in preferences + preferences_partner:
        col = [row[pref] for row in dic]
        print("Mean of {}: {}".format(pref, round(mean(col), 2)))
        



In [7]:
with open('data/dating-full.csv', newline='') as csvfile:
    fulldata = csv.DictReader(csvfile, delimiter=',')
    unq_count, lower_count, dicdata = preprocess(fulldata)

    with open('data/dating.csv', 'w') as writefile:
        writer = csv.DictWriter(writefile, dicdata[0].keys())
        writer.writeheader()
        writer.writerows(dicdata)

In [29]:
import pandas as pd

def one_hot_encoding(df):
    for cat in encoding_cols:
        vals = sorted(df[cat].unique())
        n = len(vals)

        for idx, v in enumerate(vals):
            vec = [0 for x in range(n-1)]
            if not idx == n-1:
                vec[idx] = 1
            for i in  df.index[(df.index[df[cat] == v])] :
                df.at[i, cat] = vec

            if(cat=='gender' and v=='female') or (cat=='race' and v=='Black/African American') or \
                (cat=='race_o' and v=='Other') or (cat=='field' and v=='economics'):
                print("Mapped vector for {} in column {}: {}".format(v, cat, vec))
    




In [30]:
data = pd.read_csv('data/dating.csv')

# FIXME: the handout asks to drop the last column, we are dropping the first
# hot_encoded = pd.get_dummies(data, columns = encoding_cols, drop_first=True)
# hot_encoded.to_csv('data/dating-hot.csv')

h = one_hot_encoding(data)


Mapped vector for female in column gender: [1]
Mapped vector for Black/African American in column race: [0, 1, 0, 0]
Mapped vector for Other in column race_o: [0, 0, 0, 0]
Mapped vector for economics in column field: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
